# Project Panopticon: Intelligent Exam Proctoring

**Purpose:** Demonstrate a machine-learning workflow that combines video telemetry and system-event logs to identify observations that may warrant human review.

> **Important:** This notebook is an educational prototype. A model flag is not proof of misconduct. Any alert must be reviewed by a human proctor. Results depend on the quality and representativeness of the supplied data.

## 1. Project workflow

1. Load video telemetry and system-event CSV files.
2. Validate columns and convert timestamps.
3. Handle missing sensor readings without deleting timeline rows.
4. Align asynchronous system events with video telemetry.
5. Engineer 10-observation rolling averages.
6. Prepare model features and target labels.
7. Train a class-imbalance-aware Random Forest classifier.
8. Evaluate predictions and feature importance.
9. Apply a 90% probability threshold and plot the Precision–Recall curve.

### Input files

Place these two files in the **same folder** as this notebook:

- `video telemetry.csv`
- `system events.csv`

Expected video columns: `student_id`, `exam_id`, `timestamp`, `gaze_deviation`, `head_pose_angle`, `audio_level`, `suspicious`.

Expected event columns: `student_id`, `exam_id`, `timestamp`, `event_type`, `event_value`.

## 2. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    precision_recall_curve, average_precision_score
)

RANDOM_STATE = 42
print("Libraries imported successfully.")

## 3. Load the datasets

In [ ]:
video_path = Path("video telemetry.csv")
events_path = Path("system events.csv")

if not video_path.exists() or not events_path.exists():
    raise FileNotFoundError(
        "Place 'video telemetry.csv' and 'system events.csv' in the same "
        "folder as this notebook, then run this cell again."
    )

video = pd.read_csv(video_path)
events = pd.read_csv(events_path)

print("Video dataset shape:", video.shape)
print("System events shape:", events.shape)
display(video.head())
display(events.head())

## 4. Validate and prepare columns

In [ ]:
required_video = {
    "student_id", "exam_id", "timestamp", "gaze_deviation",
    "head_pose_angle", "audio_level", "suspicious"
}
required_events = {
    "student_id", "exam_id", "timestamp", "event_type", "event_value"
}

missing_video = required_video - set(video.columns)
missing_events = required_events - set(events.columns)

if missing_video:
    raise ValueError(f"Missing columns in video telemetry: {sorted(missing_video)}")
if missing_events:
    raise ValueError(f"Missing columns in system events: {sorted(missing_events)}")

video["timestamp"] = pd.to_datetime(video["timestamp"], errors="coerce")
events["timestamp"] = pd.to_datetime(events["timestamp"], errors="coerce")

video = video.dropna(subset=["timestamp"]).copy()
events = events.dropna(subset=["timestamp"]).copy()

video["suspicious"] = pd.to_numeric(video["suspicious"], errors="coerce")
video = video.dropna(subset=["suspicious"]).copy()
video["suspicious"] = video["suspicious"].astype(int)

if not set(video["suspicious"].unique()).issubset({0, 1}):
    raise ValueError("The 'suspicious' target must contain only 0 and 1.")

print("Columns and timestamps validated.")

## 5. Handle missing video telemetry

In [ ]:
sensor_columns = ["gaze_deviation", "head_pose_angle", "audio_level"]

for column in sensor_columns:
    video[column] = pd.to_numeric(video[column], errors="coerce")

video = video.sort_values(["student_id", "timestamp"]).copy()

for column in sensor_columns:
    video[column] = video.groupby("student_id")[column].transform(
        lambda series: series.interpolate(
            method="linear", limit_direction="both"
        )
    )

print("Missing sensor values after interpolation:")
display(video[sensor_columns].isna().sum())
print("Video rows retained:", len(video))

## 6. Align system events with video telemetry

`merge_asof()` matches each video observation to the nearest event for the same student and exam, within a one-second tolerance. Event fields remain empty when no event is close enough.

In [ ]:
events_for_merge = events.rename(columns={"timestamp": "event_timestamp"}).copy()

# merge_asof requires sorting by the merge key.
video_sorted = video.sort_values("timestamp").copy()
events_sorted = events_for_merge.sort_values("event_timestamp").copy()

merged_data = pd.merge_asof(
    video_sorted,
    events_sorted,
    left_on="timestamp",
    right_on="event_timestamp",
    by=["student_id", "exam_id"],
    direction="nearest",
    tolerance=pd.Timedelta("1s")
)

merged_data["event_type"] = merged_data["event_type"].fillna("no_event")
merged_data["event_value"] = merged_data["event_value"].fillna("no_event")

print("Merged dataset shape:", merged_data.shape)
print("Matched event rows:", merged_data["event_timestamp"].notna().sum())
display(merged_data.head())

## 7. Feature engineering: 10-observation rolling averages

In [ ]:
merged_data = merged_data.sort_values(["student_id", "timestamp"]).copy()

# The source telemetry is expected to be recorded once per second.
# Therefore, 10 observations represent approximately 10 seconds.
merged_data["gaze_rolling_avg"] = (
    merged_data.groupby("student_id")["gaze_deviation"]
    .transform(lambda s: s.rolling(window=10, min_periods=1).mean())
)
merged_data["audio_rolling_avg"] = (
    merged_data.groupby("student_id")["audio_level"]
    .transform(lambda s: s.rolling(window=10, min_periods=1).mean())
)

display(merged_data[
    ["student_id", "timestamp", "gaze_deviation", "gaze_rolling_avg",
     "audio_level", "audio_rolling_avg"]
].head(12))

## 8. Explore target class distribution

In [ ]:
print("Class counts:")
display(merged_data["suspicious"].value_counts().sort_index())

ax = merged_data["suspicious"].value_counts().sort_index().plot(
    kind="bar", figsize=(6, 4), rot=0
)
ax.set_xticks([0, 1], labels=["Normal (0)", "Suspicious (1)"])
ax.set_xlabel("Activity class")
ax.set_ylabel("Number of observations")
ax.set_title("Class Distribution")
plt.tight_layout()
plt.show()

## 9. Prepare features and target

The target column `suspicious` is excluded from the input features. Student IDs, exam IDs, and timestamps are also excluded from the model inputs.

In [ ]:
numeric_features = [
    "gaze_deviation", "head_pose_angle", "audio_level",
    "gaze_rolling_avg", "audio_rolling_avg"
]

X = merged_data[numeric_features + ["event_type"]].copy()
X["event_type"] = X["event_type"].fillna("no_event").astype(str)
X = pd.get_dummies(X, columns=["event_type"], dtype=int)

y = merged_data["suspicious"].astype(int)

# Ensure all model inputs are numeric and missing values are handled.
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median(numeric_only=True)).fillna(0)

print("Feature matrix:", X.shape)
print("Target vector:", y.shape)
print("Target classes:", sorted(y.unique()))

## 10. Split the data and train the model

This baseline uses a stratified random split. Because observations from the same students/session may be related, a random split can produce optimistic results. For a stronger evaluation, use a separate exam session or hold out entire students when enough students are available.

In [ ]:
if y.nunique() < 2:
    raise ValueError("The target must contain both classes (0 and 1).")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=RANDOM_STATE
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("Model training and prediction completed.")

## 11. Evaluate the baseline model

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

baseline_results = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Score": [accuracy, precision, recall, f1]
})
display(baseline_results)

print(classification_report(
    y_test, y_pred,
    labels=[0, 1],
    target_names=["Normal Activity", "Suspicious Activity"],
    zero_division=0
))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
plt.figure(figsize=(6, 4))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Normal", "Suspicious"],
    yticklabels=["Normal", "Suspicious"]
)
plt.xlabel("Predicted activity")
plt.ylabel("Actual activity")
plt.title("Confusion Matrix - Default Threshold")
plt.tight_layout()
plt.show()

## 12. Inspect feature importance

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
}).sort_values("Importance", ascending=False)

display(feature_importance)

plt.figure(figsize=(9, 5))
plt.barh(feature_importance["Feature"], feature_importance["Importance"])
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Random Forest Feature Importance")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 13. Apply the 90% probability threshold

`predict_proba()` returns estimated class probabilities. An observation is flagged only when its estimated probability for class 1 is at least 0.90.

**A 0.90 cutoff does not guarantee 90% precision.** Precision must be measured on the evaluation data.

In [ ]:
y_prob = model.predict_proba(X_test)[:, 1]

threshold = 0.90
y_custom_pred = (y_prob >= threshold).astype(int)

threshold_results = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Score": [
        accuracy_score(y_test, y_custom_pred),
        precision_score(y_test, y_custom_pred, zero_division=0),
        recall_score(y_test, y_custom_pred, zero_division=0),
        f1_score(y_test, y_custom_pred, zero_division=0)
    ]
})
display(threshold_results)

print(classification_report(
    y_test, y_custom_pred,
    labels=[0, 1],
    target_names=["Normal Activity", "Suspicious Activity"],
    zero_division=0
))

## 14. Precision–Recall curve

In [ ]:
precision_values, recall_values, pr_thresholds = precision_recall_curve(
    y_test, y_prob
)
avg_precision = average_precision_score(y_test, y_prob)

plt.figure(figsize=(8, 5))
plt.plot(recall_values, precision_values, label=f"Average precision = {avg_precision:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall Curve")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

print("Average Precision Score:", round(avg_precision, 4))

## 15. Confusion matrix at the 90% threshold

In [ ]:
cm_90 = confusion_matrix(y_test, y_custom_pred, labels=[0, 1])

plt.figure(figsize=(6, 4))
sns.heatmap(
    cm_90, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Normal", "Suspicious"],
    yticklabels=["Normal", "Suspicious"]
)
plt.xlabel("Predicted activity")
plt.ylabel("Actual activity")
plt.title("Confusion Matrix - 90% Probability Threshold")
plt.tight_layout()
plt.show()

## 16. Review sample predictions

In [ ]:
prediction_review = pd.DataFrame({
    "Actual": y_test.to_numpy(),
    "Default Prediction": y_pred,
    "Suspicious Probability": y_prob,
    "Prediction at 90% Threshold": y_custom_pred
}, index=y_test.index)

display(prediction_review.head(20))

## 17. Conclusion

This notebook demonstrates how video telemetry and system-event logs can be combined, cleaned, and transformed into features for a baseline classification model. A Random Forest classifier is trained with class weighting, evaluated with standard classification metrics, and assessed using a Precision–Recall curve. A 90% probability cutoff is then applied to create a high-confidence review queue.

### Limitations and responsible use

- This is a prototype, not a validated production proctoring system.
- Model performance depends on the quality, size, and representativeness of the data.
- A high probability is not proof of cheating or misconduct.
- False positives and false negatives must be considered.
- Human review, transparency, privacy protections, and an appeal process are essential.
- For reliable evaluation, test on separate students and exam sessions not used during training.